<a href="https://colab.research.google.com/github/MahderAkalu/Comparative-Evaluation-of-Privacy-Preserving-Techniques-in-Deep-Federated-Learning/blob/master/Shakespeare0423.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args])

# 2. Core ML stack

pip("install", "-q", "numpy==1.26.4")


# PyTorch

try:
    import torch
    print("PyTorch already installed:", torch.__version__)
except:
    pip("install", "-q", "torch", "torchvision")

pip("install", "-q",
    "flwr[simulation]",
    "opacus",
    "scikit-learn",
    "matplotlib",
    "pandas",
    "tqdm",
    "requests"
)

print(" INSTALL COMPLETE — restart runtime if needed")

In [ ]:
# =============================================================
# Federated NLP – Comparative Privacy Evaluation on Shakespeare
#   R0  = FedAvg (No Privacy)
#   R1a = DP-FL  sigma = 0.5
#   R1b = DP-FL  sigma = 1.0
#   R2  = SecAgg (simulated)
#
# Dataset     : Shakespeare next-character prediction
# Model       : LSTM Language Model
# Partitioner : Dirichlet  alpha = 0.5  (non-IID)
# Seeds       : 42, 123, 7
# Metrics     : Accuracy, Perplexity, MIA-AUC,
#               Communication Cost, Privacy-Utility Trade-off
# =============================================================

import os, random, string, warnings, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import flwr as fl
from opacus import PrivacyEngine
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import requests
warnings.filterwarnings("ignore")

print(f"flwr  : {fl.__version__}")
print(f"torch : {torch.__version__}")
print(f"numpy : {np.__version__}")


#  GLOBAL CONSTANTS                                  

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

SEEDS           = [42, 123, 7]
NUM_CLIENTS     = 10
NUM_ROUNDS      = 10
FRAC_FIT        = 0.5
BATCH_SIZE      = 64
LOCAL_EPOCHS    = 1
SEQ_LEN         = 30
EMBED_DIM       = 64
HIDDEN_DIM      = 128
NUM_LAYERS      = 1
CLIP_NORM       = 1.0
LR              = 3e-3
DIRICHLET_ALPHA = 0.5

ALL_CHARS  = string.printable
VOCAB_SIZE = len(ALL_CHARS)
stoi = {ch: i for i, ch in enumerate(ALL_CHARS)}
itos = {i: ch for ch, i in stoi.items()}

def encode(txt):
    return [stoi[c] for c in txt if c in stoi]


# DOWNLOAD SHAKESPEARE CORPUS                       

print("Downloading Shakespeare ...")
URL       = ("https://raw.githubusercontent.com/karpathy/"
             "char-rnn/master/data/tinyshakespeare/input.txt")
_text     = requests.get(URL).text
print(f"  {len(_text):,} characters downloaded.")

_split    = int(0.9 * len(_text))
TRAIN_IDS = encode(_text[:_split])
TEST_IDS  = encode(_text[_split:])


# DIRICHLET PARTITIONER                         

def dirichlet_partition(ids, n_clients, alpha, seed):
   
    rng    = np.random.default_rng(seed)
    arr    = np.array(ids)
    N      = len(arr)
    props  = rng.dirichlet(np.full(n_clients, alpha))
    cuts   = (np.cumsum(props[:-1]) * N).astype(int)
    chunks = np.split(arr, cuts)
    rng.shuffle(chunks)
    return chunks


#  DATASET CLASS + LOADERS                           

class ShakespeareDataset(Dataset):
    def __init__(self, ids):
        self.ids = list(ids)

    def __len__(self):
        return max(1, len(self.ids) - SEQ_LEN - 1)

    def __getitem__(self, idx):
        x = torch.tensor(self.ids[idx     : idx + SEQ_LEN],     dtype=torch.long)
        y = torch.tensor(self.ids[idx + 1 : idx + SEQ_LEN + 1], dtype=torch.long)
        return x, y


def make_loaders(client_chunks, cid):
    ids    = client_chunks[cid]
    sp     = int(0.85 * len(ids))
    tr_ldr = DataLoader(ShakespeareDataset(ids[:sp]),
                        batch_size=BATCH_SIZE, shuffle=True,
                        drop_last=True,  num_workers=0)
    va_ldr = DataLoader(ShakespeareDataset(ids[sp:]),
                        batch_size=BATCH_SIZE, shuffle=False,
                        drop_last=False, num_workers=0)
    return tr_ldr, va_ldr


GLOBAL_TESTLOADER = DataLoader(
    ShakespeareDataset(TEST_IDS),
    batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=0,
)
print("Dataset ready.")


# LSTM MODEL                                        

class LSTMNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, EMBED_DIM)
        self.lstm  = nn.LSTM(EMBED_DIM, HIDDEN_DIM, NUM_LAYERS,
                             batch_first=True)
        self.fc    = nn.Linear(HIDDEN_DIM, VOCAB_SIZE)

    def forward(self, x):
        out, _ = self.lstm(self.embed(x))
        return self.fc(out)


def get_parameters(net):
    return [v.cpu().numpy() for v in net.state_dict().values()]

def set_parameters(net, parameters):
    sd = {k: torch.tensor(v)
          for k, v in zip(net.state_dict().keys(), parameters)}
    net.load_state_dict(sd, strict=True)


# TRAIN / EVAL                      

def train_one_round(net, loader, dp_enabled=False,
                    noise_multiplier=0.0, clip_norm=1.0):
    net.to(DEVICE).train()
    crit = nn.CrossEntropyLoss()
    opt  = torch.optim.Adam(net.parameters(), lr=LR)

    if dp_enabled and len(loader) > 0:
        pe = PrivacyEngine()
        net, opt, loader = pe.make_private(
            module=net, optimizer=opt, data_loader=loader,
            noise_multiplier=noise_multiplier, max_grad_norm=clip_norm,
        )

    for _ in range(LOCAL_EPOCHS):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            crit(net(x).reshape(-1, VOCAB_SIZE),
                 y.reshape(-1)).backward()
            opt.step()

    comm = sum(p.numel() * 4 for p in net.parameters())
    return net, len(loader.dataset), comm


@torch.no_grad()
def evaluate_model(net, loader):
    net.to(DEVICE).eval()
    crit = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out   = net(x)
        total_loss += crit(out.reshape(-1, VOCAB_SIZE), y.reshape(-1)).item()
        correct    += (out.argmax(-1) == y).sum().item()
        total      += y.numel()
    avg = total_loss / max(len(loader), 1)
    return avg, correct / max(total, 1), float(np.exp(avg))


@torch.no_grad()
def compute_mia_auc(net, member_ldr, nonmember_ldr, cap=300):

    net.to(DEVICE).eval()
    crit = nn.CrossEntropyLoss(reduction='none')

    def collect(ldr, label, limit):
        losses, labs = [], []
        for x, y in ldr:
            x, y = x.to(DEVICE), y.to(DEVICE)
            per  = (crit(net(x).reshape(-1, VOCAB_SIZE), y.reshape(-1))
                    .reshape(x.size(0), -1).mean(1))
            losses.extend(per.cpu().tolist())
            labs.extend([label] * x.size(0))
            if len(losses) >= limit:
                break
        return losses[:limit], labs[:limit]

    ml, mlab = collect(member_ldr,    1, cap)
    nl, nlab = collect(nonmember_ldr, 0, cap)
    try:
        return roc_auc_score(mlab + nlab, [-s for s in ml + nl])
    except Exception:
        return 0.5

# SECAGG STRATEGY (SIMULATED)                

class SecAggFedAvg(fl.server.strategy.FedAvg):
   
    def __init__(self, noise_std=0.005, **kwargs):
        super().__init__(**kwargs)
        self.noise_std = noise_std

    def aggregate_fit(self, server_round, results, failures):
        agg = super().aggregate_fit(server_round, results, failures)
        if agg is None:
            return agg
        params, metrics = agg
        arrays = fl.common.parameters_to_ndarrays(params)
        noisy  = [a + np.random.normal(0, self.noise_std, a.shape)
                  for a in arrays]
        return fl.common.ndarrays_to_parameters(noisy), metrics

# FLOWER CLIENT & SERVER EVALUATE FN              

_CLIENT_CHUNKS = []
_EXP_CFG       = {}
_ROUND_METRICS = []


class ShakespeareClient(fl.client.NumPyClient):
    def __init__(self, cid):
        self.cid = int(cid)
        self.net = LSTMNet().to(DEVICE)
        self.tr_ldr, self.va_ldr = make_loaders(_CLIENT_CHUNKS, self.cid)

    def get_parameters(self, config):
        return get_parameters(self.net)

    def fit(self, parameters, config):
        set_parameters(self.net, parameters)
        cfg = _EXP_CFG
        net, n, comm = train_one_round(
            self.net, self.tr_ldr,
            cfg["dp_enabled"], cfg["noise_multiplier"], cfg["clip_norm"],
        )
        return get_parameters(net), n, {"comm_bytes": float(comm)}

    def evaluate(self, parameters, config):
        set_parameters(self.net, parameters)
        loss, acc, ppl = evaluate_model(self.net, self.va_ldr)
        return loss, len(self.va_ldr.dataset), {
            "accuracy": acc, "perplexity": ppl,
        }


def client_fn(cid):
    return ShakespeareClient(int(cid))


def make_evaluate_fn(exp_name):
    def evaluate_fn(server_round, parameters, config):
        net = LSTMNet()
        set_parameters(net, parameters)
        loss, acc, ppl = evaluate_model(net, GLOBAL_TESTLOADER)
        tr0, _ = make_loaders(_CLIENT_CHUNKS, 0)
        mia    = compute_mia_auc(net, tr0, GLOBAL_TESTLOADER)
        print(f"  [{exp_name}] Rnd {server_round:02d} | "
              f"Acc={acc:.4f}  PPL={ppl:.2f}  MIA={mia:.4f}")
        _ROUND_METRICS.append({"round": server_round, "loss": loss,
                                "accuracy": acc, "perplexity": ppl,
                                "mia_auc": mia})
        return loss, {"accuracy": acc, "perplexity": ppl, "mia_auc": mia}
    return evaluate_fn

# SINGLE-SEED RUNNER                               

def run_one_seed(exp_name, seed,
                 dp_enabled=False, noise_multiplier=0.0,
                 clip_norm=CLIP_NORM, use_secagg=False):
    global _CLIENT_CHUNKS, _EXP_CFG, _ROUND_METRICS

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    _CLIENT_CHUNKS = dirichlet_partition(
        TRAIN_IDS, NUM_CLIENTS, DIRICHLET_ALPHA, seed=seed,
    )
    sizes = [len(c) for c in _CLIENT_CHUNKS]
    print(f"    Partition | min={min(sizes):,}  max={max(sizes):,}  "
          f"mean={int(np.mean(sizes)):,} tokens")

    _EXP_CFG = dict(dp_enabled=dp_enabled,
                    noise_multiplier=noise_multiplier,
                    clip_norm=clip_norm)
    _ROUND_METRICS = []

    torch.manual_seed(0)
    init_p = fl.common.ndarrays_to_parameters(get_parameters(LSTMNet()))

    common_kw = dict(
        fraction_fit          = FRAC_FIT,
        fraction_evaluate     = 0.0,
        min_fit_clients       = max(1, int(NUM_CLIENTS * FRAC_FIT)),
        min_available_clients = NUM_CLIENTS,
        on_fit_config_fn      = lambda rnd: _EXP_CFG,
        evaluate_fn           = make_evaluate_fn(exp_name),
        initial_parameters    = init_p,
    )

    strategy = (SecAggFedAvg(noise_std=0.005, **common_kw)
                if use_secagg
                else fl.server.strategy.FedAvg(**common_kw))

    t0 = time.time()
    fl.simulation.start_simulation(
        client_fn     = client_fn,
        num_clients   = NUM_CLIENTS,
        config        = fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy      = strategy,
        ray_init_args = {"ignore_reinit_error": True,
                         "num_cpus": os.cpu_count()},
    )
    print(f"    Done in {time.time()-t0:.0f}s  (seed={seed})")


    pb   = sum(p.numel() * 4 for p in LSTMNet().parameters())
    comm = pb * NUM_ROUNDS * int(NUM_CLIENTS * FRAC_FIT)

    return {"seed": seed, "metrics": list(_ROUND_METRICS),
            "comm_total_MB": comm / 1e6}

#  MULTI-SEED EXPERIMENT WRAPPER                    

def run_experiment(exp_name, **kwargs):
    print(f"\n{'='*62}")
    print(f"  EXPERIMENT : {exp_name}")
    print(f"  Seeds      : {SEEDS}  |  Dirichlet alpha={DIRICHLET_ALPHA}")
    print(f"{'='*62}")

    seed_results = [run_one_seed(exp_name, s, **kwargs) for s in SEEDS]

    rounds = [m["round"] for m in seed_results[0]["metrics"]]
    agg    = {"rounds": rounds}

    for metric in ("accuracy", "perplexity", "mia_auc", "loss"):
        mat = np.array([[m[metric] for m in sr["metrics"]]
                        for sr in seed_results])
        agg[f"{metric}_mean"] = mat.mean(0).tolist()
        agg[f"{metric}_std"]  = mat.std(0).tolist()

    comms = [sr["comm_total_MB"] for sr in seed_results]
    agg["comm_mean_MB"] = float(np.mean(comms))
    agg["comm_std_MB"]  = float(np.std(comms))

    print(f"\n  Summary | "
          f"Acc={agg['accuracy_mean'][-1]:.4f}+/-{agg['accuracy_std'][-1]:.4f}  "
          f"PPL={agg['perplexity_mean'][-1]:.2f}  "
          f"MIA={agg['mia_auc_mean'][-1]:.4f}  "
          f"Comm={agg['comm_mean_MB']:.1f} MB")

    return {"name": exp_name, "agg": agg, "seed_results": seed_results}

# EXPERIMENTS                              

all_results = {}

all_results["R0"] = run_experiment(
    "R0: FedAvg (No Privacy)",
    dp_enabled=False,
)

all_results["R1a"] = run_experiment(
    "R1a: DP-FL sigma=0.5",
    dp_enabled=True, noise_multiplier=0.5, clip_norm=CLIP_NORM,
)

all_results["R1b"] = run_experiment(
    "R1b: DP-FL sigma=1.0",
    dp_enabled=True, noise_multiplier=1.0, clip_norm=CLIP_NORM,
)

all_results["R2"] = run_experiment(
    "R2: SecAgg",
    dp_enabled=False, use_secagg=True,
)

print("\n\n*** ALL EXPERIMENTS COMPLETE ***")

#  PLOT HELPERS                              

COLORS = {"R0": "#2196F3", "R1a": "#4CAF50",
          "R1b": "#FF6F00", "R2":  "#9C27B0"}
KEYS   = ["R0", "R1a", "R1b", "R2"]


def shade_plot(ax, metric, ylabel, title, baseline=None):
    for key in KEYS:
        a    = all_results[key]["agg"]
        rnds = a["rounds"]
        mean = np.array(a[f"{metric}_mean"])
        std  = np.array(a[f"{metric}_std"])
        c    = COLORS[key]
        ax.plot(rnds, mean, marker='o', markersize=3, lw=1.8,
                color=c, label=all_results[key]["name"])
        ax.fill_between(rnds, mean - std, mean + std,
                        alpha=0.15, color=c)
    if baseline is not None:
        ax.axhline(baseline, ls='--', color='grey', lw=1,
                   label=f"Baseline ({baseline})")
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel("Communication Round", fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.25)
    ax.tick_params(labelsize=8)

# FULL PANEL FIGURE                                        

fig = plt.figure(figsize=(18, 15))
fig.suptitle(
    "Federated NLP – Comparative Privacy Evaluation\n"
    f"Shakespeare | LSTM | 10 Clients | 10 Rounds | "
    f"Dirichlet alpha={DIRICHLET_ALPHA} | Seeds {SEEDS}",
    fontsize=14, fontweight='bold', y=0.99,
)
gs   = gridspec.GridSpec(3, 2, figure=fig, hspace=0.46, wspace=0.32)
axes = [fig.add_subplot(gs[r, c]) for r in range(3) for c in range(2)]

# Panel 1 Accuracy
shade_plot(axes[0], "accuracy", "Accuracy (higher is better)",
           "Test Accuracy  (mean +/- 1 std)")

# Panel 2 Perplexity
shade_plot(axes[1], "perplexity", "Perplexity (lower is better)",
           "Test Perplexity  (mean +/- 1 std)")

# Panel 3 MIA AUC
shade_plot(axes[2], "mia_auc",
           "MIA AUC  (0.5 = fully private)",
           "Membership Inference AUC  (mean +/- 1 std)",
           baseline=0.5)

# Panel 4 Communication cost
ax4   = axes[3]
means = [all_results[k]["agg"]["comm_mean_MB"] for k in KEYS]
stds  = [all_results[k]["agg"]["comm_std_MB"]  for k in KEYS]
xlabs = [all_results[k]["name"].replace(": ", "\n") for k in KEYS]
x     = np.arange(len(KEYS))
bars  = ax4.bar(x, means, yerr=stds, capsize=5, width=0.55,
                color=[COLORS[k] for k in KEYS],
                error_kw={"elinewidth": 1.5})
ax4.bar_label(bars, labels=[f"{v:.1f}" for v in means],
              fontsize=8, padding=4)
ax4.set_title("Total Communication Cost  (mean +/- std)", fontsize=11,
              fontweight='bold')
ax4.set_ylabel("MB transferred", fontsize=9)
ax4.set_xticks(x); ax4.set_xticklabels(xlabs, fontsize=8)
ax4.grid(True, axis='y', alpha=0.25)
ax4.tick_params(labelsize=8)

# Panel 5 Privacy-Utility scatter
ax5 = axes[4]
for key in KEYS:
    a = all_results[key]["agg"]
    c = COLORS[key]
    for sr in all_results[key]["seed_results"]:
        fin = sr["metrics"][-1] if sr["metrics"] else {}
        ax5.scatter(fin.get("mia_auc", 0.5), fin.get("accuracy", 0),
                    s=35, color=c, alpha=0.45, zorder=2)
    mx = a["mia_auc_mean"][-1]
    my = a["accuracy_mean"][-1]
    ax5.scatter(mx, my, s=150, color=c, zorder=4,
                edgecolors='black', linewidths=0.7,
                label=all_results[key]["name"])
    ax5.annotate(key, (mx, my), textcoords="offset points",
                 xytext=(6, 4), fontsize=8)
ax5.axvline(0.5, ls='--', color='grey', lw=1, label="MIA baseline")
ax5.set_xlabel("MIA AUC  (lower = more private)", fontsize=9)
ax5.set_ylabel("Accuracy  (higher = more useful)", fontsize=9)
ax5.set_title("Privacy-Utility Trade-off  (Final Round, all seeds)",
              fontsize=11, fontweight='bold')
ax5.legend(fontsize=7); ax5.grid(True, alpha=0.25)
ax5.tick_params(labelsize=8)

# Panel 6 – Summary table
ax6 = axes[5]
ax6.axis('off')
col_labels = ["Method", "Accuracy (mean+/-std)", "PPL",
              "MIA AUC", "Comm MB (mean+/-std)"]
rows = []
for key in KEYS:
    a     = all_results[key]["agg"]
    short = all_results[key]["name"].split(":")[1].strip()
    rows.append([
        short,
        f"{a['accuracy_mean'][-1]:.4f} +/- {a['accuracy_std'][-1]:.4f}",
        f"{a['perplexity_mean'][-1]:.2f}",
        f"{a['mia_auc_mean'][-1]:.4f}",
        f"{a['comm_mean_MB']:.1f} +/- {a['comm_std_MB']:.1f}",
    ])
tbl = ax6.table(cellText=rows, colLabels=col_labels,
                loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1.15, 1.9)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1A237E')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#E8EAF6')
    cell.set_edgecolor('#BDBDBD')
ax6.set_title("Final-Round Summary  (mean +/- std across 3 seeds)",
              fontsize=11, fontweight='bold', pad=14)

plt.savefig("federated_nlp_results.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved to federated_nlp_results.png")